In [1]:
import rospy
from grasp_planner.curobo_grasp_planner import GraspPlanner
from sensor_msgs.msg import JointState
from perception.perception_interface import PerceptionInterface
from task_planner.motoman import MotomanSDA10F
from motion_planner.curobo_planner import CuroboPlanner
from task_planner.eutils import execute
from typing import Optional
from cv_bridge import CvBridge

robot: Optional[MotomanSDA10F] = None
perception: Optional[PerceptionInterface] = None
planner: Optional[CuroboPlanner] = None  
grasp_planner: Optional[GraspPlanner] = None
bridge: Optional[CvBridge] = None

def init_robot():
    global robot, perception, planner, grasp_planner, bridge
    rospy.init_node("jupyter_controller")
    bridge = CvBridge()

    robot = MotomanSDA10F(is_sim=True, ground_truth=True)
    perception = robot.init_perception_interface()
    planner = robot.init_motion_planner(planner="curobo")  # ,warmup=False)
    grasp_planner = GraspPlanner(
        robot.curobo_config,
        planner.static_world_config,
        robot.urdf,
        ignore_collision_ee_links=robot.ignore_collision_ee_links,
    )

init_robot()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TORCH_COMPILE is not set, Disabling.
Environment variable for CUROBO_TO

trunc margin 0.01
Voxel volume size: 250 x 640 x 248 - # points: 39,680,000
camera0/color/camera_info
camera1/color/camera_info


Unknown tag "material" in /robot[@name='motoman_sda10f']/link[@name='left_inner_finger_pad']/collision[1]
Unknown tag "material" in /robot[@name='motoman_sda10f']/link[@name='right_inner_finger_pad']/collision[1]
Unknown tag "material" in /robot[@name='motoman_sda10f']/link[@name='left_inner_finger_pad']/collision[1]
Unknown tag "material" in /robot[@name='motoman_sda10f']/link[@name='right_inner_finger_pad']/collision[1]


Warp UserWarning: Python 3.9 or newer is recommended for running Warp, detected sys.version_info(major=3, minor=8, micro=10, releaselevel='final', serial=0)
tensor([[-2.2689, -2.9496, -2.9496, -2.9496, -2.9496, -3.4732, -3.4732, -6.9639],
        [ 2.2689,  2.9496,  2.9496,  2.9496,  2.9496,  3.4732,  3.4732,  6.9639]],
       device='cuda:0')
tensor([[-0.5672, -0.7374, -0.7374, -0.7374, -0.7374, -0.8683, -0.8683, -1.7410],
        [ 0.5672,  0.7374,  0.7374,  0.7374,  0.7374,  0.8683,  0.8683,  1.7410]],
       device='cuda:0')


In [ ]:
import sys
import task_planner.eutils as eutils

DEFAULT_TARGET_OVERRIDES = {
    "torso_joint_b1": 0.0,
    "arm_right_joint_1_s": 0.2,
    "arm_right_joint_2_l": -0.7,
    "arm_right_joint_3_e": 0.0,
    "arm_right_joint_4_u": -1.7,
    "arm_right_joint_5_r": 0.0,
    "arm_right_joint_6_b": -1.3,
    "arm_right_joint_7_t": 0.0,
}

ROBOT_RESET_JOINTS = {
    "torso_joint_b1": 0,
    "arm_left_joint_1_s": 1.75,
    "arm_left_joint_2_l": 0.8,
    "arm_left_joint_3_e": 0,
    "arm_left_joint_4_u": -0.66,
    "arm_left_joint_5_r": 0,
    "arm_left_joint_6_b": 0,
    "arm_left_joint_7_t": 0,
    "arm_right_joint_1_s": 0.2,
    "arm_right_joint_2_l": -0.7,
    "arm_right_joint_3_e": 0.0,
    "arm_right_joint_4_u": -1.7,
    "arm_right_joint_5_r": 0,
    "arm_right_joint_6_b": -1.3,
    "arm_right_joint_7_t": 0.0,
}

ROBOT_DROP=[0.1, -0.7, 1.2, 0.0, 1.0, 0.0, 0.0]
ROBOT_PREGRASP=[0.65, 0.05, 1.3, 0.0, 1.0, 0.0, 0.0]

def build_target(planner, current_map, overrides):
    missing = [n for n in planner.motion_gen.joint_names if n not in current_map]
    if missing:
        raise KeyError(f"Missing joints in /joint_states_all: {missing}")

    target = {name: current_map[name] for name in planner.motion_gen.joint_names}
    target.update(overrides)
    return target

def plan():
    # print("PLANNER: ", planner)
    joint_state = rospy.wait_for_message("/joint_states_all", JointState, timeout=5)
    current_map = dict(zip(joint_state.name, joint_state.position))

    plan, result = planner.pose_motion_plan(
        joint_state,
        ROBOT_PREGRASP,
        visualize=True,
        return_all=True,
    )

    if plan is None:
        print(f"Plan failed: {result.status}", file=sys.stderr)
        return 1

    print("Plan succeeded.")
    execute(plan, window=0, wait=True)
    print("STOPPED MOVING")
    # res = eutils.get_experiment_result(0)
    # print("RESULT: ", res)

plan()

using grasp pose
Plan succeeded.
Waiting for robot to stop...
Stopped!
Moving!
Waiting for robot to stop...
Stopped!
STOPPED MOVING
